In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
import joblib
import time

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')

# Remove problematic and cow-specific features
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'disturbance', 'cow.1'])

# Separate features and targets
X = df.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Features used: {list(X.columns)}")

# 2. Create LightGBM model with MultiOutputClassifier
model = lgb.LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
multi_output_model = MultiOutputClassifier(model)

# 3. Define parameter grid
param_grid = {
    'estimator__num_leaves': [31, 63],
    'estimator__max_depth': [-1, 10, 20],  # -1 means no limit
    'estimator__learning_rate': [0.05, 0.1],
    'estimator__n_estimators': [200, 300],
    'estimator__reg_alpha': [0, 0.1],
    'estimator__reg_lambda': [0, 0.1],
    'estimator__scale_pos_weight': [None, 'balanced'],
    'estimator__subsample': [0.8, 1.0],
}

# 4. Custom cross-validation for multi-output stratification
def multilabel_stratified_cv(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    dummy = y.iloc[:, 0]
    return skf.split(X, dummy)

# 5. Create and run GridSearchCV
print("Starting hyperparameter tuning with GridSearchCV...")
start_time = time.time()

search = GridSearchCV(
    multi_output_model,
    param_grid=param_grid,
    cv=multilabel_stratified_cv(X_train, y_train, n_splits=3),
    scoring='roc_auc_ovr_weighted',
    refit=True,
    verbose=3,
    n_jobs=-1,
    error_score='raise'
)

search.fit(X_train, y_train)

print(f"Hyperparameter tuning completed in {time.time()-start_time:.2f} seconds")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.4f}")

# 6. Evaluate on test set
print("\nEvaluating on test set...")
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)

# Classification report for each disease
diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
test_results = {}

for i, disease in enumerate(diseases):
    print(f"\n{'='*50}\n{disease.upper()} PERFORMANCE\n{'='*50}")
    print(classification_report(y_test[disease], y_pred[:, i]))
    test_results[disease] = {
        'auc': roc_auc_score(y_test[disease], y_proba[i][:, 1])
    }

# 7. Feature Importance Analysis
print("\nFeature Importances:")
feature_importances = []
for idx, disease in enumerate(diseases):
    importances = search.best_estimator_.estimators_[idx].feature_importances_
    feature_importances.append(importances)

importance_df = pd.DataFrame(
    np.array(feature_importances).T,
    columns=diseases,
    index=X.columns
)
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("\nTop Features:")
print(importance_df)

# 8. Save the best model and results
joblib.dump(search.best_estimator_, 'lightgbm_multioutput_model.joblib')
importance_df.to_csv('lightgbm_feature_importances.csv')
print("Model and feature importances saved")

# 9. Final summary
print("\n" + "="*50)
print("FINAL MODEL SUMMARY")
print("="*50)
print(f"Best parameters: {search.best_params_}")
print(f"Cross-validation AUC: {search.best_score_:.4f}")

for disease in diseases:
    print(f"{disease.upper()} Test AUC: {test_results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 8), Test shape: (6576, 8)
Features used: ['hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
Starting hyperparameter tuning with GridSearchCV...
Fitting 3 folds for each of 384 candidates, totalling 1152 fits


LightGBMError: Unknown token balanced in data file

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
import joblib
import time
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.validation import check_is_fitted

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'disturbance', 'cow.1'])
X = df.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Features used: {list(X.columns)}")

# 2. Calculate scale_pos_weight for each disease
scale_pos_weights = {}
for disease in y.columns:
    pos_count = y_train[disease].sum()
    neg_count = len(y_train) - pos_count
    scale_pos_weights[disease] = neg_count / pos_count if pos_count > 0 else 1.0
    print(f"{disease}: scale_pos_weight = {scale_pos_weights[disease]:.2f}")

# 3. Create improved custom classifier
class CustomMultiOutputClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator, weights):
        self.estimator = estimator
        self.weights = weights
        
    def fit(self, X, y, **fit_params):
        self.estimators_ = [clone(self.estimator) for _ in range(y.shape[1])]
        self.classes_ = []  # Store classes for each output
        
        # Set weights and fit each estimator
        for i, estimator in enumerate(self.estimators_):
            disease = y.columns[i]
            estimator.set_params(scale_pos_weight=self.weights[disease])
            estimator.fit(X, y.iloc[:, i], **fit_params)
            
            # Store classes for this estimator
            self.classes_.append(estimator.classes_)
            
        return self
    
    def predict(self, X):
        check_is_fitted(self)
        predictions = np.column_stack([
            estimator.predict(X) for estimator in self.estimators_
        ])
        return predictions
    
    def predict_proba(self, X):
        check_is_fitted(self)
        probas = [estimator.predict_proba(X) for estimator in self.estimators_]
        return probas

# 4. Create LightGBM model with custom multi-output classifier
base_model = lgb.LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
multi_output_model = CustomMultiOutputClassifier(base_model, scale_pos_weights)

# 5. Define simplified parameter grid
param_dist = {
    'estimator__num_leaves': [31, 63],
    'estimator__max_depth': [-1, 10],
    'estimator__learning_rate': [0.05, 0.1],
    'estimator__n_estimators': [200, 300],
    'estimator__reg_alpha': [0, 0.1],
    'estimator__reg_lambda': [0, 0.1],
}

# 6. Custom cross-validation
def multilabel_stratified_cv(X, y, n_splits=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    dummy = y.iloc[:, 0]
    return skf.split(X, dummy)

# 7. Create and run RandomizedSearchCV
print("Starting hyperparameter tuning with RandomizedSearchCV...")
start_time = time.time()

search = RandomizedSearchCV(
    multi_output_model,
    param_distributions=param_dist,
    n_iter=10,  # Reduced to 10 combinations for faster execution
    cv=multilabel_stratified_cv(X_train, y_train, n_splits=3),
    scoring='roc_auc_ovr_weighted',
    refit=True,
    random_state=42,
    verbose=3,
    n_jobs=-1,
    error_score='raise'
)

search.fit(X_train, y_train)

print(f"Hyperparameter tuning completed in {time.time()-start_time:.2f} seconds")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.4f}")

# 8. Evaluate on test set
print("\nEvaluating on test set...")
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)

diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
test_results = {}

for i, disease in enumerate(diseases):
    print(f"\n{'='*50}\n{disease.upper()} PERFORMANCE\n{'='*50}")
    print(classification_report(y_test[disease], y_pred[:, i]))
    test_results[disease] = {
        'auc': roc_auc_score(y_test[disease], y_proba[i][:, 1])
    }

# 9. Feature Importance Analysis
print("\nFeature Importances:")
feature_importances = []
for idx, disease in enumerate(diseases):
    importances = search.best_estimator_.estimators_[idx].feature_importances_
    feature_importances.append(importances)

importance_df = pd.DataFrame(
    np.array(feature_importances).T,
    columns=diseases,
    index=X.columns
)
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("\nTop Features:")
print(importance_df)

# 10. Save the best model and results
joblib.dump(search.best_estimator_, 'lightgbm_multioutput_model.joblib')
importance_df.to_csv('lightgbm_feature_importances.csv')
print("Model and feature importances saved")

# 11. Final summary
print("\n" + "="*50)
print("FINAL MODEL SUMMARY")
print("="*50)
print(f"Best parameters: {search.best_params_}")
print(f"Cross-validation AUC: {search.best_score_:.4f}")

for disease in diseases:
    print(f"{disease.upper()} Test AUC: {test_results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 8), Test shape: (6576, 8)
Features used: ['hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
oestrus: scale_pos_weight = 3.14
calving: scale_pos_weight = 6.76
lameness: scale_pos_weight = 9.31
mastitis: scale_pos_weight = 29.98
Starting hyperparameter tuning with RandomizedSearchCV...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Hyperparameter tuning completed in 260.94 seconds
Best parameters: {'estimator__reg_lambda': 0, 'estimator__reg_alpha': 0, 'estimator__num_leaves': 31, 'estimator__n_estimators': 200, 'estimator__max_depth': 10, 'estimator__learning_rate': 0.05}
Best CV score: 0.6923

Evaluating on test set...

OESTRUS PERFORMANCE
              precision    recall  f1-score   support

           0       0.82      0.69      0.75      4987
           1       0.36      0.53      0.43      1589

    accuracy                           0.65      6576
   macro avg       0

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
import joblib
import time
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'disturbance'])

# Feature engineering
df['cow_id'] = LabelEncoder().fit_transform(df['cow'])
df['activity_rest_ratio'] = df['ACTIVITY_LEVEL'] / (df['REST'] + 1e-6)

# Separate features and targets
X = df[['cow_id', 'hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
        'rest_to_eat_ratio', 'activity_rest_ratio']]
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

# Split into train/test with cow-aware splitting
cow_groups = df['cow_id']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# 2. Train individual models per disease
diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
models = {}
results = {}

for disease in diseases:
    print(f"\n{'='*50}\nTraining {disease.upper()} model\n{'='*50}")
    
    # Calculate class weights
    pos_count = y_train[disease].sum()
    neg_count = len(y_train) - pos_count
    scale_pos_weight = neg_count / pos_count
    print(f"Class ratio: 1:{scale_pos_weight:.1f}")
    
    # Create model
    model = lgb.LGBMClassifier(
        num_leaves=63,
        max_depth=7,
        learning_rate=0.01,
        n_estimators=1000,
        reg_alpha=0.1,
        reg_lambda=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    # Train with early stopping
    model.fit(
        X_train, y_train[disease],
        eval_set=[(X_test, y_test[disease])],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    # Evaluate
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test[disease], y_pred))
    auc = roc_auc_score(y_test[disease], y_proba)
    print(f"AUC: {auc:.4f}")
    
    # Store results
    models[disease] = model
    results[disease] = {
        'classification_report': classification_report(y_test[disease], y_pred, output_dict=True),
        'auc': auc
    }

# 3. Feature importance analysis
print("\nFeature Importances:")
for disease, model in models.items():
    importance = pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)
    
    print(f"\n{disease.upper()}:")
    print(importance.head(10))

# 4. Save models
for disease, model in models.items():
    joblib.dump(model, f'{disease}_model.joblib')
print("All models saved")

# 5. Final summary
print("\n" + "="*50)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*50)
for disease in diseases:
    print(f"{disease.upper()} AUC: {results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 8), Test shape: (6576, 8)

Training OESTRUS model
Class ratio: 1:3.1
              precision    recall  f1-score   support

           0       0.81      0.94      0.87      4987
           1       0.63      0.31      0.41      1589

    accuracy                           0.79      6576
   macro avg       0.72      0.63      0.64      6576
weighted avg       0.77      0.79      0.76      6576

AUC: 0.7744

Training CALVING model
Class ratio: 1:6.8


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

              precision    recall  f1-score   support

           0       0.86      1.00      0.93      5668
           1       0.00      0.00      0.00       908

    accuracy                           0.86      6576
   macro avg       0.43      0.50      0.46      6576
weighted avg       0.74      0.86      0.80      6576

AUC: 0.8747

Training LAMENESS model
Class ratio: 1:9.3


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

              precision    recall  f1-score   support

           0       0.90      1.00      0.95      5912
           1       0.00      0.00      0.00       664

    accuracy                           0.90      6576
   macro avg       0.45      0.50      0.47      6576
weighted avg       0.81      0.90      0.85      6576

AUC: 0.8111

Training MASTITIS model
Class ratio: 1:30.0
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      6369
           1       0.00      0.00      0.00       207

    accuracy                           0.97      6576
   macro avg       0.48      0.50      0.49      6576
weighted avg       0.94      0.97      0.95      6576

AUC: 0.8582

Feature Importances:

OESTRUS:
cow_id                 2323
IN_ALLEYS               612
hour                    585
REST                    261
ACTIVITY_LEVEL          251
activity_rest_ratio     234
rest_to_eat_ratio       164
EAT                     158
dtype: int32

CALVING

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag